In [1]:
!pip install transformers datasets torch scikit-learn pandas numpy accelerate --quiet

In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast
from datasets import Dataset
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
from transformers import (
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DistilBertTokenizerFast
)
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split



In [3]:
import re
import pandas as pd

# File paths (update if needed)
ANOMALY_FILE = r'C:\Users\jatin\Desktop\dem\new\workspace\projects\transformers\server.log'
NORMAL_FILE = r'C:\Users\jatin\Desktop\dem\new\workspace\projects\transformers\2server.log.2024-10-08-12'

LOG_START_RE = re.compile(r'^\[\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2},\d+\]')

LOG_PARSE_RE = re.compile(
    r'^\[(?P<timestamp>[^\]]+)\]'        # timestamp
    r'\s+(?P<level>\w+)'                 # log level (INFO, ERROR, etc.)
    r'(?:\s+\[(?P<component>[^\]]+)\])?' # optional component
    r'\s*(?P<message>.+)$',              # message
    re.DOTALL
)

print('Regex patterns defined')

Regex patterns defined


In [4]:
def read_log_file(filepath: str) -> list:
    """
    Reads a log file and combines multi-line entries into single lines.
    Returns a list of complete log entries.
    """
    entries = []
    current_entry = []

    with open(filepath, 'r', encoding='utf-8', errors='replace') as file:
        for line in file:
            line = line.rstrip('\n')

            # If a new log entry starts, save the previous one
            if LOG_START_RE.match(line):
                if current_entry:
                    entries.append(' '.join(current_entry))
                current_entry = [line]
            else:
                # Continue the current log entry (e.g., stack traces)
                if line.strip():
                    current_entry.append(line.strip())

    # Add the last entry if it exists
    if current_entry:
        entries.append(' '.join(current_entry))

    return entries


def parse_log_entry(raw: str) -> dict:
    """
    Parses a log entry into timestamp, level, component, and message.
    """
    match = LOG_PARSE_RE.match(raw)

    if match:
        return {
            'raw': raw,
            'timestamp': match.group('timestamp').strip(),
            'level': match.group('level').strip().upper(),
            'component': match.group('component').strip() if match.group('component') else '',
            'message': match.group('message').strip(),
        }

    # If parsing fails, return a fallback structure
    return {
        'raw': raw,
        'timestamp': '',
        'level': 'UNKNOWN',
        'component': '',
        'message': raw,
    }



In [5]:
# Load both files
print('Loading anomaly log file...')
anomaly_raw = read_log_file(ANOMALY_FILE)
print(f'Loaded {len(anomaly_raw)} entries from anomaly file')

print('Loading normal log file...')
normal_raw = read_log_file(NORMAL_FILE)
print(f'Loaded {len(normal_raw)} entries from normal file')

# Parse logs into structured records
anomaly_records = [parse_log_entry(entry) for entry in anomaly_raw]
normal_records = [parse_log_entry(entry) for entry in normal_raw]

# Add labels (1 = anomaly, 0 = normal)
for record in anomaly_records:
    record['label'] = 1

for record in normal_records:
    record['label'] = 0

# Combine into a single DataFrame
df = pd.DataFrame(anomaly_records + normal_records)

# Shuffle the data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nData loaded successfully. Total rows: {len(df)}')

Loading anomaly log file...
Loaded 704 entries from anomaly file
Loading normal log file...
Loaded 117 entries from normal file

Data loaded successfully. Total rows: 821


In [6]:
print('First 10 rows:\n')

pd.set_option('display.max_colwidth', 80)
display(df[['timestamp', 'level', 'component', 'message', 'label']].head(10))
# Dataset statistics
print('=' * 50)
print('Dataset Statistics')
print('=' * 50)

print(f'Total entries: {len(df)}')
print(f'Normal (0): {(df.label == 0).sum()}')
print(f'Anomaly (1): {(df.label == 1).sum()}')

print('\nLog level breakdown:')
level_counts = df['level'].value_counts()
for level, count in level_counts.items():
    print(f'{level:10s}: {count}')

print('\nNull or empty values:')
print(df[['timestamp', 'level', 'component', 'message']].isnull().sum().to_string())

print('=' * 50)
print('Data looks good. Proceed to next step: Cleaning and Preprocessing')

First 10 rows:



,timestamp,level,component,message,label
0,"2024-10-08 16:07:22,712",INFO,GroupMetadataManager brokerId=1,Finished loading offsets and group metadata from __consumer_offsets-39 in 2 ...,1
1,"2024-10-08 16:07:22,027",WARN,"ReplicaFetcher replicaId=1, leaderId=2, fetcherId=0",Partition _confluent-metrics-3 marked as failed (kafka.server.ReplicaFetcher...,1
2,"2024-10-08 16:07:13,278",INFO,"ReplicaFetcher replicaId=1, leaderId=0, fetcherId=0",Node 0 disconnected. (org.apache.kafka.clients.NetworkClient),1
3,"2024-10-08 16:07:22,023",WARN,"ReplicaFetcher replicaId=1, leaderId=2, fetcherId=0",Partition _confluent-controlcenter-7-3-9-1-actual-group-consumption-rekey-0 ...,1
4,"2024-10-08 16:07:22,564",INFO,"ReplicaFetcher replicaId=1, leaderId=2, fetcherId=0",Truncating partition _confluent-controlcenter-7-3-9-1-Group-ONE_MINUTE-chang...,1
5,"2024-10-08 16:07:22,567",INFO,"ReplicaFetcher replicaId=1, leaderId=2, fetcherId=0",Truncating partition _confluent-controlcenter-7-3-9-1-MonitoringMessageAggre...,1
6,"2024-10-08 12:49:59,400",INFO,"LocalLog partition=_confluent-telemetry-metrics-0, dir=/opt/kafka-logs","Deleting segment files LogSegment(baseOffset=3432062, size=3533118, lastModi...",0
7,"2024-10-08 16:07:15,485",INFO,Producer clientId=confluent-metrics-reporter,Node 0 disconnected. (org.apache.kafka.clients.NetworkClient),1
8,"2024-10-08 16:07:22,683",INFO,MergedLog partition=_confluent-controlcenter-7-3-9-1-AlertHistoryStore-repar...,Truncating to 0 has no effect as the largest offset in the log is -1 (kafka....,1
9,"2024-10-08 12:23:59,403",INFO,"MergedLog partition=_confluent-telemetry-metrics-2, dir=/opt/kafka-logs","Deleting segment LogSegment(baseOffset=3386571, size=3477509, lastModifiedTi...",0


Dataset Statistics
Total entries: 821
Normal (0): 117
Anomaly (1): 704

Log level breakdown:
INFO      : 740
WARN      : 81

Null or empty values:
timestamp    0
level        0
component    0
message      0
Data looks good. Proceed to next step: Cleaning and Preprocessing


In [7]:
# Spot check: view one normal and one anomaly entry
print('Sample NORMAL entry:\n')

sample_normal = df[df.label == 0].iloc[0]
print(f'Timestamp : {sample_normal["timestamp"]}')
print(f'Level     : {sample_normal["level"]}')
print(f'Component : {sample_normal["component"]}')
print(f'Message   : {sample_normal["message"][:200]}')

print('\nSample ANOMALY entry:\n')

sample_anomaly = df[df.label == 1].iloc[0]
print(f'Timestamp : {sample_anomaly["timestamp"]}')
print(f'Level     : {sample_anomaly["level"]}')
print(f'Component : {sample_anomaly["component"]}')
print(f'Message   : {sample_anomaly["message"][:200]}')
# Save the parsed DataFrame
save_path = 'kafka_logs_parsed.csv'
df.to_csv(save_path, index=False)

print(f'Parsed data saved to: {save_path}')
print('You can open this file in Excel to review the data if needed.')

Sample NORMAL entry:

Timestamp : 2024-10-08 12:49:59,400
Level     : INFO
Component : LocalLog partition=_confluent-telemetry-metrics-0, dir=/opt/kafka-logs
Message   : Deleting segment files LogSegment(baseOffset=3432062, size=3533118, lastModifiedTime=1728112461731, largestRecordTimestamp=Some(1728112461734)) (kafka.log.LocalLog$)

Sample ANOMALY entry:

Timestamp : 2024-10-08 16:07:22,712
Level     : INFO
Component : GroupMetadataManager brokerId=1
Message   : Finished loading offsets and group metadata from __consumer_offsets-39 in 2 milliseconds for epoch 5, of which 2 milliseconds was spent in the scheduler. (kafka.coordinator.group.GroupMetadataManager)
Parsed data saved to: kafka_logs_parsed.csv
You can open this file in Excel to review the data if needed.


In [8]:

# Load parsed data
df = pd.read_csv('kafka_logs_parsed.csv')

print('Loaded kafka_logs_parsed.csv')
print(f'Rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print(f'Normal: {(df.label == 0).sum()}')
print(f'Anomaly: {(df.label == 1).sum()}')


# Define cleaning patterns

# 13-digit timestamps (epoch time)
EPOCH_TS_RE = re.compile(r'\b\d{13}\b')

# File paths like /opt/kafka-logs/topic/00000.log
FILEPATH_RE = re.compile(r'/[\w./-]+')

# Java class references like (kafka.log.X)
JAVA_CLASS_RE = re.compile(r'\([a-z][\w.]+\$?\)')

# Stack trace lines like: at kafka.server.X.method(File.scala:99)
STACKTRACE_RE = re.compile(r'at\s+[\w.$]+\([\w.]+:\d+\)')

# Hex values like 0x1a2b
HEX_RE = re.compile(r'0x[0-9a-fA-F]+')

# Long numbers (IDs, offsets)
LONG_NUM_RE = re.compile(r'\b\d{7,}\b')

# Extra whitespace
WHITESPACE_RE = re.compile(r'\s+')


def clean_message(text: str) -> str:
    """
    Cleans a log message by removing noise while keeping useful information.
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    text = STACKTRACE_RE.sub(' ', text)
    text = JAVA_CLASS_RE.sub(' ', text)
    text = FILEPATH_RE.sub(' ', text)
    text = EPOCH_TS_RE.sub(' ', text)
    text = HEX_RE.sub(' ', text)
    text = LONG_NUM_RE.sub('NUM', text)
    text = WHITESPACE_RE.sub(' ', text)

    return text.strip()


print('Cleaning function ready\n')

# Quick test
sample = df['message'].iloc[0]

print(f'Before: {sample[:200]}')
print()
print(f'After : {clean_message(sample)[:200]}')

Loaded kafka_logs_parsed.csv
Rows: 821
Columns: ['raw', 'timestamp', 'level', 'component', 'message', 'label']
Normal: 117
Anomaly: 704
Cleaning function ready

Before: Finished loading offsets and group metadata from __consumer_offsets-39 in 2 milliseconds for epoch 5, of which 2 milliseconds was spent in the scheduler. (kafka.coordinator.group.GroupMetadataManager)

After : Finished loading offsets and group metadata from __consumer_offsets-39 in 2 milliseconds for epoch 5, of which 2 milliseconds was spent in the scheduler.


In [9]:
# Apply cleaning
df['text'] = df['message'].apply(clean_message)

# Remove empty rows after cleaning
before = len(df)
df = df[df['text'].str.strip().str.len() > 0].reset_index(drop=True)
dropped = before - len(df)

print('Cleaning applied to all rows')
print(f'Rows before: {before}')
print(f'Rows dropped: {dropped}')
print(f'Rows after: {len(df)}\n')

# Compare before vs after (first 3 rows)
print('Before vs After (first 3 rows):\n')
for i in range(min(3, len(df))):
    print(f'[{i}] BEFORE: {df["message"].iloc[i][:120]}')
    print(f'[{i}] AFTER : {df["text"].iloc[i][:120]}\n')


# Check text length distribution
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
df['approx_tokens'] = (df['word_count'] * 1.3).astype(int)

print('Text length statistics (approx tokens):')
print(df['approx_tokens'].describe().round(1).to_string())
print()

# Coverage for different max_length values
for max_len in [64, 128, 256, 512]:
    pct = (df['approx_tokens'] <= max_len).mean() * 100
    print(f'max_length={max_len:4d} -> covers {pct:.1f}% of entries')

print('\nRecommendation: choose the smallest max_length covering ~95% of entries')

Cleaning applied to all rows
Rows before: 821
Rows dropped: 0
Rows after: 821

Before vs After (first 3 rows):

[0] BEFORE: Finished loading offsets and group metadata from __consumer_offsets-39 in 2 milliseconds for epoch 5, of which 2 millise
[0] AFTER : Finished loading offsets and group metadata from __consumer_offsets-39 in 2 milliseconds for epoch 5, of which 2 millise

[1] BEFORE: Partition _confluent-metrics-3 marked as failed (kafka.server.ReplicaFetcherThread)
[1] AFTER : Partition _confluent-metrics-3 marked as failed

[2] BEFORE: Node 0 disconnected. (org.apache.kafka.clients.NetworkClient)
[2] AFTER : Node 0 disconnected.

Text length statistics (approx tokens):
count     821.0
mean       64.5
std       297.6
min         1.0
25%        11.0
50%        15.0
75%        19.0
max      1913.0

max_length=  64 -> covers 96.8% of entries
max_length= 128 -> covers 96.8% of entries
max_length= 256 -> covers 97.1% of entries
max_length= 512 -> covers 97.3% of entries

Recommendation

In [10]:
# Preview cleaned dataset
print('Final cleaned dataset (text and label):\n')

pd.set_option('display.max_colwidth', 100)
display(df[['text', 'label']].head(10))

print('\nLabel distribution:')
print(df['label'].value_counts().rename({0: 'Normal (0)', 1: 'Anomaly (1)'}).to_string())


# Save cleaned dataset
clean_path = 'kafka_logs_clean.csv'
df[['text', 'label']].to_csv(clean_path, index=False)

print(f'\nCleaned dataset saved to: {clean_path}')
print(f'Columns: text, label')
print(f'Total rows: {len(df)}')

print('\nReady for next step: Tokenization')

Final cleaned dataset (text and label):



,text,label
0,Finished loading offsets and group metadata from __consumer_offsets-39 in 2 milliseconds for epo...,1
1,Partition _confluent-metrics-3 marked as failed,1
2,Node 0 disconnected.,1
3,Partition _confluent-controlcenter-7-3-9-1-actual-group-consumption-rekey-0 marked as failed,1
4,Truncating partition _confluent-controlcenter-7-3-9-1-Group-ONE_MINUTE-changelog-3 with Truncati...,1
5,Truncating partition _confluent-controlcenter-7-3-9-1-MonitoringMessageAggregatorWindows-ONE_MIN...,1
6,"Deleting segment files LogSegment(baseOffset=NUM, size=NUM, lastModifiedTime= , largestRecordTim...",0
7,Node 0 disconnected.,1
8,Truncating to 0 has no effect as the largest offset in the log is -1,1
9,"Deleting segment LogSegment(baseOffset=NUM, size=NUM, lastModifiedTime= , largestRecordTimestamp...",0



Label distribution:
label
Anomaly (1)    704
Normal (0)     117

Cleaned dataset saved to: kafka_logs_clean.csv
Columns: text, label
Total rows: 821

Ready for next step: Tokenization


In [11]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast
from datasets import Dataset

# Config
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256
TEST_SIZE = 0.2
RANDOM_SEED = 42

# Load cleaned data
df = pd.read_csv('kafka_logs_clean.csv')

print('Loaded kafka_logs_clean.csv')
print(f'Total rows: {len(df)}')
print(f'Normal (0): {(df.label == 0).sum()}')
print(f'Anomaly (1): {(df.label == 1).sum()}')
print(f'max_length: {MAX_LENGTH}')


# Train / validation split (stratified)
train_df, val_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print('\nSplit complete')
print(f'Train: {len(train_df)} rows -> Normal={(train_df.label == 0).sum()} Anomaly={(train_df.label == 1).sum()}')
print(f'Val  : {len(val_df)} rows -> Normal={(val_df.label == 0).sum()} Anomaly={(val_df.label == 1).sum()}')

Loaded kafka_logs_clean.csv
Total rows: 821
Normal (0): 117
Anomaly (1): 704
max_length: 256

Split complete
Train: 656 rows -> Normal=93 Anomaly=563
Val  : 165 rows -> Normal=24 Anomaly=141


In [12]:
# Load tokenizer
print(f'Loading tokenizer: {MODEL_NAME}...')
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

print('Tokenizer loaded')
print(f'Vocab size: {tokenizer.vocab_size:,}')
print(f'Max model length: {tokenizer.model_max_length}')
print(f'Using max length: {MAX_LENGTH}\n')

# Quick test
sample_text = train_df['text'].iloc[0]
sample_tokens = tokenizer(sample_text, truncation=True, max_length=MAX_LENGTH)

print('Sample tokenization:')
print(f'Text   : {sample_text[:100]}')
print(f'Tokens : {len(sample_tokens["input_ids"])}')
print(f'IDs    : {sample_tokens["input_ids"][:10]} ...')


# Tokenize and convert to Hugging Face Dataset
def tokenize_dataframe(df: pd.DataFrame) -> Dataset:
    """Convert DataFrame to tokenized Hugging Face Dataset."""

    dataset = Dataset.from_pandas(
        df.rename(columns={'label': 'labels'})
    )

    def tokenize_fn(batch):
        return tokenizer(
            batch['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH,
        )

    dataset = dataset.map(tokenize_fn, batched=True, batch_size=64)
    dataset = dataset.remove_columns(['text'])
    dataset.set_format('torch')

    return dataset


print('\nTokenizing train set...')
train_dataset = tokenize_dataframe(train_df)
print(f'Train samples: {len(train_dataset)}')

print('\nTokenizing validation set...')
val_dataset = tokenize_dataframe(val_df)
print(f'Validation samples: {len(val_dataset)}')

Loading tokenizer: distilbert-base-uncased...
Tokenizer loaded
Vocab size: 30,522
Max model length: 512
Using max length: 256

Sample tokenization:
Text   : Error in response for fetch request (type=FetchRequest, replicaId=1, maxWait=500, minBytes=1, maxByt
Tokens : 256
IDs    : [101, 7561, 1999, 3433, 2005, 18584, 5227, 1006, 2828, 1027] ...

Tokenizing train set...


Map:   0%|          | 0/656 [00:00<?, ? examples/s]

Train samples: 656

Tokenizing validation set...


Map:   0%|          | 0/165 [00:00<?, ? examples/s]

Validation samples: 165


In [13]:
# Verify one tokenized sample
sample = train_dataset[0]

print('Verifying one tokenized training sample:')
print(f'Keys           : {list(sample.keys())}')
print(f'input_ids shape: {sample["input_ids"].shape}')
print(f'attention_mask : {sample["attention_mask"].shape}')
print(f'label          : {sample["labels"].item()} -> {"Normal" if sample["labels"].item() == 0 else "Anomaly"}')

print('\nTokenized datasets are ready\n')
print('train_dataset:', train_dataset)
print('val_dataset  :', val_dataset)


# Summary before training
print('=' * 50)
print('Tokenization Summary')
print('=' * 50)

print(f'Tokenizer     : {MODEL_NAME}')
print(f'max_length    : {MAX_LENGTH}')
print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')
print(f'Input shape   : {train_dataset[0]["input_ids"].shape}')
print('Format        : PyTorch tensors')

print('=' * 50)
print('\nReady for next step: Load model and train')

Verifying one tokenized training sample:
Keys           : ['labels', 'input_ids', 'attention_mask']
input_ids shape: torch.Size([256])
attention_mask : torch.Size([256])
label          : 1 -> Anomaly

Tokenized datasets are ready

train_dataset: Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 656
})
val_dataset  : Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 165
})
Tokenization Summary
Tokenizer     : distilbert-base-uncased
max_length    : 256
Train samples : 656
Val samples   : 165
Input shape   : torch.Size([256])
Format        : PyTorch tensors

Ready for next step: Load model and train


In [14]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
from transformers import (
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DistilBertTokenizerFast
)
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_fp16 = torch.cuda.is_available()

print(f'Device: {device}')
print(f'FP16 enabled: {use_fp16}')


# Rebuild datasets (safe to rerun)
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256
RANDOM_SEED = 42

df = pd.read_csv('kafka_logs_clean.csv')

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)


def tokenize_dataframe(df):
    dataset = Dataset.from_pandas(df.rename(columns={'label': 'labels'}))

    def tokenize_fn(batch):
        return tokenizer(
            batch['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH
        )

    dataset = dataset.map(tokenize_fn, batched=True, batch_size=64)
    dataset = dataset.remove_columns(['text'])
    dataset.set_format('torch')

    return dataset


train_dataset = tokenize_dataframe(train_df)
val_dataset = tokenize_dataframe(val_df)

print(f'Datasets ready: train={len(train_dataset)}, val={len(val_dataset)}')

Device: cpu
FP16 enabled: False


Map:   0%|          | 0/656 [00:00<?, ? examples/s]

Map:   0%|          | 0/165 [00:00<?, ? examples/s]

Datasets ready: train=656, val=165


In [15]:
# Compute class weights (to handle imbalance)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print('Class weights computed:')
print(f'Normal (0)  : {class_weights[0]:.4f}')
print(f'Anomaly (1) : {class_weights[1]:.4f}\n')

if class_weights[1] > class_weights[0]:
    print('Anomaly class has higher weight, so the model will focus more on detecting anomalies')
else:
    print('Classes are roughly balanced')

Class weights computed:
Normal (0)  : 3.5269
Anomaly (1) : 0.5826

Classes are roughly balanced


In [16]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, and F1 score."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='binary',
        pos_label=1,
        zero_division=0
    )

    macro_f1 = precision_recall_fscore_support(
        labels,
        preds,
        average='macro',
        zero_division=0
    )[2]

    return {
        'accuracy': round(float(acc), 4),
        'precision_anomaly': round(float(precision), 4),
        'recall_anomaly': round(float(recall), 4),
        'f1_anomaly': round(float(f1), 4),
        'f1_macro': round(float(macro_f1), 4),
    }


# Custom trainer with weighted loss
class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


print('compute_metrics() defined')
print('WeightedTrainer defined')

compute_metrics() defined
WeightedTrainer defined


In [17]:
# Load model
print(f'Loading model: {MODEL_NAME}...')

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'Normal', 1: 'Anomaly'},
    label2id={'Normal': 0, 'Anomaly': 1},
)

model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model loaded on {device}')
print(f'Total parameters: {total:,}')
print(f'Trainable parameters: {trainable:,}')

Loading model: distilbert-base-uncased...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\jatin\Desktop\dem\new\workspace\projects\transformers\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jatin\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cpu
Total parameters: 66,955,010
Trainable parameters: 66,955,010


In [18]:
# Load DistilBERT model
print(f'Loading model: {MODEL_NAME}...')

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'Normal', 1: 'Anomaly'},
    label2id={'Normal': 0, 'Anomaly': 1},
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model loaded on {device}')
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')


# Training arguments
training_args = TrainingArguments(
    output_dir='./kafka_model',

    # Training setup
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Optimizer
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    # Evaluation and saving
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='recall_anomaly',
    greater_is_better=True,

    # Logging
    logging_dir='./kafka_model/logs',
    logging_steps=10,
    report_to='none',

    # Performance
    fp16=use_fp16,
    dataloader_num_workers=0,
    seed=RANDOM_SEED,
)

print('TrainingArguments configured')
print('Epochs            : 5')
print('Batch size (train): 16')
print('Learning rate     : 2e-5')
print('Best model metric : recall_anomaly')
print('FP16 enabled      :', use_fp16)

Loading model: distilbert-base-uncased...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Model loaded on cpu
Total parameters    : 66,955,010
Trainable parameters: 66,955,010
TrainingArguments configured
Epochs            : 5
Batch size (train): 16
Learning rate     : 2e-5
Best model metric : recall_anomaly
FP16 enabled      : False


In [19]:
# Build trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Trainer ready')
print('\nStarting training...')
print('Training will stop early if recall_anomaly stops improving\n')

train_result = trainer.train()

print('\nTraining complete')
print(f'Training loss : {train_result.training_loss:.4f}')
print(f'Runtime       : {train_result.metrics["train_runtime"]:.1f}s')
print(f'Samples/sec   : {train_result.metrics["train_samples_per_second"]:.1f}')


Trainer ready

Starting training...
Training will stop early if recall_anomaly stops improving



c:\Users\jatin\Desktop\dem\new\workspace\projects\transformers\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Anomaly,Recall Anomaly,F1 Anomaly,F1 Macro
1,0.263578,0.152120,0.981800,1.000000,0.978700,0.989200,0.965200
2,0.169734,0.067195,0.981800,1.000000,0.978700,0.989200,0.965200
3,0.187936,0.072074,0.981800,1.000000,0.978700,0.989200,0.965200
4,0.122756,0.064298,0.981800,1.000000,0.978700,0.989200,0.965200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\jatin\Desktop\dem\new\workspace\projects\transformers\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\jatin\Desktop\dem\new\workspace\projects\transformers\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training complete
Training loss : 0.2257
Runtime       : 1273.4s
Samples/sec   : 2.6
